In [1]:
import pandas as pd
import os
import sys


In [2]:
%pwd

'a:\\Future\\MLOPS\\Projects\\Data science pro\\research'

In [3]:
os.chdir("../")
%pwd

'a:\\Future\\MLOPS\\Projects\\Data science pro'

In [ ]:
import os
import dagshub
import mlflow
# ← Get from DagsHub Settings

# ✅ Step 2 — Then init
dagshub.init(repo_owner="Praveenravva61", repo_name="Data-science-pro", mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/Praveenravva61/Data-science-pro.mlflow")

a:\Future\MLOPS\Projects\Data science pro\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as Praveenravva61

Initialized MLflow to track repo "Praveenravva61/Data-science-pro"

Repository Praveenravva61/Data-science-pro initialized!

In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    metric_filename: Path
    all_params: dict
    target_column: str
    mlflow_uri: str
    
    

In [7]:
from src.DataSciencePro.constants import *
from src.DataSciencePro.utils.common import *


class ConfigurationManager:
    def __init__(
        self,
        config_file_path= CONFIG_FILE_PATH,
        params_file_path= PARAM_FILE_PATH,
        schema_file_path= SCHEME_FILE_PATH):
        
        self.config= read_yaml(config_file_path)
        self.params= read_yaml(params_file_path)
        self.schema= read_yaml(schema_file_path)
        
        create_directories([self.config.artifacts_root])
       
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation   
        params = self.params.ElasticNet
        schema= self.schema.TARGET_COLUMN
        
        create_directories([config.root_dir])
        
        
        model_evaluation_config= ModelEvaluationConfig(
            root_dir= config.root_dir,
            test_data_path= config.test_data_path,
            model_path= config.model_path,
            metric_filename= config.metric_filename,
            all_params= params, 
            target_column= schema.name,
            mlflow_uri= "https://dagshub.com/Praveenravva61/Data-science-pro.mlflow"
        )     
        return model_evaluation_config
    
    

In [8]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib



In [9]:
import dagshub

In [10]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config= config
        
    def evaluation(self, actual, pred):
        rmse= np.sqrt(mean_squared_error(actual, pred))
        mae= mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    def log_into_mlfow(self):
        
        dagshub.init(
        repo_owner='Praveenravva61',
        repo_name='Data-science-pro',
        mlflow=True
        )

        test_data= pd.read_csv(self.config.test_data_path)
        model= joblib.load(self.config.model_path)
        
        
        test_x = test_data.drop([self.config.target_column], axis= 1)
        test_y= test_data[[self.config.target_column]]
        
        
        mlflow.set_tracking_uri(self.config.mlflow_uri)

        mlflow_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)
            rmse, mae, r2 = self.evaluation(test_y, predicted_qualities)
        
            mlflow.log_params(self.config.all_params)
            mlflow.log_metric('rmse', rmse)
            mlflow.log_metric('mae', mae)
            mlflow.log_metric('r2_score', r2)
        
            try:
                if mlflow_url_type_store != 'file':
                    mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticNet")
                else:
                    mlflow.sklearn.log_model(model, "model")
                print("✅ Model artifact saved successfully")
            except Exception as e:
                print(f"❌ log_model failed: {e}")  # ← Check what error appears here
                
                
            

In [ ]:
try:
    config= ConfigurationManager()
    get_model_eval_config= config.get_model_evaluation_config()
    model_evaluation= ModelEvaluation(config = get_model_eval_config)
    model_evaluation.log_into_mlfow()
except Exception as e:
    raise e
    

[2026-05-05 17:03:41,855 - DatascienceLogger - INFO - yaml file: config\config.yaml is loaded sucessfully.]
[2026-05-05 17:03:41,858 - DatascienceLogger - INFO - yaml file: params.yaml is loaded sucessfully.]
[2026-05-05 17:03:41,862 - DatascienceLogger - INFO - yaml file: schema.yaml is loaded sucessfully.]
Created directory at: artifacts
Created directory at: artifacts/Model_Evaluation
[2026-05-05 17:03:43,194 - httpx - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/Praveenravva61/Data-science-pro "HTTP/1.1 200 OK"]


Initialized MLflow to track repo "Praveenravva61/Data-science-pro"

[2026-05-05 17:03:43,205 - dagshub - INFO - Initialized MLflow to track repo "Praveenravva61/Data-science-pro"]


Repository Praveenravva61/Data-science-pro initialized!

[2026-05-05 17:03:43,211 - dagshub - INFO - Repository Praveenravva61/Data-science-pro initialized!]


2026/05/05 17:03:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 17:03:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ElasticNet' already exists. Creating a new version of this model...
2026/05/05 17:04:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNet, version 6
Created version '6' of model 'ElasticNet'.


✅ Model artifact saved successfully
🏃 View run merciful-mouse-321 at: https://dagshub.com/Praveenravva61/Data-science-pro.mlflow/#/experiments/0/runs/0567798d6f3b4075a945d4dc3d0c1992
🧪 View experiment at: https://dagshub.com/Praveenravva61/Data-science-pro.mlflow/#/experiments/0
